# Stable-profit selection

Pool diagnostics, the fixed severe-loss classifier and three pre-registered risk vetoes.

In [1]:
from pathlib import Path
import sys
import json
import math
import hashlib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd() / "scratchpad/hyperliquid-ic" if (Path.cwd() / "scratchpad/hyperliquid-ic").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / "_artifacts-stable-profit"
ARTIFACTS.mkdir(exist_ok=True)
REWRITE = PROJECT_DIR / "_artifacts-rewrite"
from ic_research import ResearchConfig, completed_daily_window, simulate_allocator, summarise_backtest, daily_feature_ic, write_json
from stable_profit import classification_metrics, paired_block_sharpe_samples, short_core_columns, simulate_stable_policy, walk_forward_loss_predictions, weekly_curve

CONFIG = ResearchConfig()
features = pd.read_parquet(REWRITE / "features.parquet")
labels = pd.read_parquet(REWRITE / "labels.parquet")
observations = pd.read_parquet(REWRITE / "observations.parquet")
evaluation_start, evaluation_end = completed_daily_window(observations)
features = features[features.date.between(evaluation_start - pd.Timedelta(days=CONFIG.training_history_days), evaluation_end)].copy()
labels = labels[labels.date.isin(features.date)].copy()
evaluation = features[features.date.between(evaluation_start, evaluation_end)].copy()
print("Stable-profit panel:", features.shape, "| dates:", features.date.nunique(), "| evaluation:", evaluation_start, "to", evaluation_end)


Stable-profit panel: (104618, 410) | dates: 545 | evaluation: 2025-09-13 00:00:00 to 2026-09-12 00:00:00


In [2]:
evaluation = features[features.date.between(evaluation_start, evaluation_end)].copy()
evaluation_labels = labels[labels.date.between(evaluation_start, evaluation_end)].copy()
incumbent = evaluation[(evaluation.nav_age_days <= 1) & evaluation.incumbent_return_gate.gt(-.16)].copy()
score_stats = incumbent.groupby("date").agg(pool=("address", "size"), finite_score=("incumbent_score", "count"), zero_score=("incumbent_score", lambda s: int((s.fillna(0) == 0).sum())), distinct_score=("incumbent_score", "nunique")).reset_index()
score_stats["tie_fraction"] = 1 - score_stats.distinct_score / score_stats.pool.clip(lower=1)
score_stats.to_csv(ARTIFACTS / "incumbent-score-saturation.csv", index=False)

common = evaluation.merge(evaluation_labels[["date", "address", "forward_log_growth_raw_30"]], on=["date", "address"], how="inner")
common = common[common.forward_log_growth_raw_30.notna() & common.interval_downside_dev_30.notna()].copy()
common["downside_quintile"] = common.groupby("date").interval_downside_dev_30.transform(lambda s: pd.qcut(s.rank(method="first"), 5, labels=False) + 1 if len(s) >= 5 else np.nan)
common["incumbent_neighbourhood"] = common.incumbent_score.notna() | common.incumbent_return_gate.gt(-.16)
q = common.groupby("downside_quintile", observed=False).agg(rows=("address", "size"), dates=("date", "nunique"), mean_growth=("forward_log_growth_raw_30", "mean"), loss_probability=("forward_log_growth_raw_30", lambda s: float((s < math.log(.9)).mean()))).reset_index()
q.to_csv(ARTIFACTS / "downside-quintile-forward.csv", index=False)
common[common.incumbent_neighbourhood].groupby("downside_quintile", observed=False).agg(rows=("address", "size"), mean_growth=("forward_log_growth_raw_30", "mean"), loss_probability=("forward_log_growth_raw_30", lambda s: float((s < math.log(.9)).mean()))).reset_index().to_csv(ARTIFACTS / "downside-quintile-incumbent-neighbourhood.csv", index=False)

beta = common.dropna(subset=["btc_beta_30", "interval_downside_dev_30"]).copy()
beta["risk_q"] = beta.groupby("date").interval_downside_dev_30.transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=False) + 1 if len(s) >= 3 else np.nan)
beta["abs_beta_q"] = beta.groupby("date").btc_beta_30.transform(lambda s: pd.qcut(s.abs().rank(method="first"), 3, labels=False) + 1 if len(s) >= 3 else np.nan)
beta.groupby(["risk_q", "abs_beta_q"], observed=False).agg(rows=("address", "size"), mean_growth=("forward_log_growth_raw_30", "mean"), loss_probability=("forward_log_growth_raw_30", lambda s: float((s < math.log(.9)).mean()))).reset_index().to_csv(ARTIFACTS / "risk-beta-double-sort.csv", index=False)

cohorts = evaluation.copy()
cohorts["history_cohort"] = pd.cut(cohorts.available_history_days, [-np.inf, 7, 14, 30, 90, 180, 360, np.inf], labels=["<7", "7-13", "14-29", "30-89", "90-179", "180-359", "360+"], right=False)
cohorts["sampling_cohort"] = np.select([cohorts.is_sparse_prediction.fillna(False), cohorts.last_interval_days.gt(2), cohorts.last_interval_days.gt(1)], ["weekly-plus", "irregular", "two-day"], default="daily")
cohorts.groupby(["history_cohort", "sampling_cohort"], observed=False).agg(rows=("address", "size"), dates=("date", "nunique"), vaults=("address", "nunique"), median_nav_age=("nav_age_days", "median")).reset_index().to_csv(ARTIFACTS / "age-sampling-cohorts.csv", index=False)
pd.DataFrame([{"horizon": h, "feature_rows": len(features), "raw_growth_labels": int(labels[f"forward_log_growth_raw_{h}"].notna().sum()), "downside_labels": int(labels[f"forward_downside_semivariance_{h}"].notna().sum()), "unique_outcomes": int(labels[f"outcome_id_{h}"].nunique())} for h in CONFIG.forecast_horizons]).to_csv(ARTIFACTS / "label-attrition.csv", index=False)
display(score_stats.describe())
display(q)



,date,pool,finite_score,zero_score,distinct_score,tie_fraction
count,257,257.000000,257.000000,257.000000,257.000000,257.000000
mean,2026-05-03 06:54:37.821011712,228.638132,64.136187,179.494163,38.330739,0.837332
min,2025-09-17 00:00:00,1.000000,0.000000,1.000000,0.000000,0.714859
25%,2026-03-04 00:00:00,223.000000,56.000000,164.000000,32.000000,0.816850
50%,2026-05-07 00:00:00,242.000000,64.000000,187.000000,39.000000,0.843750
75%,2026-07-10 00:00:00,254.000000,84.000000,202.000000,48.000000,0.858537
max,2026-09-12 00:00:00,278.000000,109.000000,226.000000,71.000000,1.000000
std,NaN,41.447979,23.038541,31.496915,13.352623,0.045025


,downside_quintile,rows,dates,mean_growth,loss_probability
0,1,15652,335,-0.013509,0.061270
1,2,15451,335,-0.021029,0.111061
2,3,15451,335,-0.028848,0.182254
3,4,15451,335,-0.081415,0.323992
4,5,15580,335,-0.324667,0.474326


In [3]:
core = short_core_columns(features)
predictions = walk_forward_loss_predictions(features, labels, feature_columns=core, horizon=30, evaluation_dates=pd.DatetimeIndex(sorted(evaluation.date.unique())), min_train_dates=60, c=1.0)
predictions.to_parquet(ARTIFACTS / "loss-predictions-30.parquet", index=False)
write_json(ARTIFACTS / "loss-classifier-metrics.json", classification_metrics(predictions))
calibration = predictions[(predictions.status == "predicted") & predictions.loss_probability.notna() & predictions.actual_loss.notna()].copy()
calibration["probability_bin"] = pd.qcut(calibration.loss_probability.rank(method="first"), 10, labels=False) + 1 if len(calibration) >= 10 else np.nan
calibration.groupby("probability_bin", observed=False).agg(rows=("address", "size"), mean_probability=("loss_probability", "mean"), realised_loss_rate=("actual_loss", "mean"), mean_growth=("actual", "mean")).reset_index().to_csv(ARTIFACTS / "loss-calibration.csv", index=False)
display(pd.DataFrame([classification_metrics(predictions)]))

# Compare all three veto scores on identical finite rows and retain the
# outcome-free permutation controls used by the plan.
finite_rows = common.dropna(subset=["interval_downside_dev_30", "btc_beta_30"]).merge(
    predictions[["date", "address", "loss_probability"]], on=["date", "address"], how="inner"
)
finite_rows["abs_btc_beta_30"] = finite_rows.btc_beta_30.abs()
tail_rows = []
for date, group in finite_rows.groupby("date", sort=True):
    count = math.floor(.2 * len(group))
    if count < 1:
        continue
    row = {"date": date, "rows": len(group), "excluded_rows": count}
    for name, score in [("a1", "interval_downside_dev_30"), ("a2", "loss_probability"), ("a3", "abs_btc_beta_30")]:
        selected = group.sort_values([score, "address"], ascending=[False, True], kind="stable").head(count)
        row[f"{name}_tail_growth"] = float(selected.forward_log_growth_raw_30.mean())
        row[f"{name}_tail_loss_probability"] = float((selected.forward_log_growth_raw_30 < math.log(.9)).mean())
    tail_rows.append(row)
matched = pd.DataFrame(tail_rows)
matched.to_csv(ARTIFACTS / "matched-finite-veto-rows.csv", index=False)
rng = np.random.default_rng(41)
null_rows = []
for draw in range(20):
    selected = []
    for _, group in finite_rows.groupby("date"):
        count = math.floor(.2 * len(group))
        if count:
            selected.extend(rng.choice(group.index.to_numpy(), size=count, replace=False))
    sample = finite_rows.loc[selected]
    null_rows.append({"draw": draw, "excluded_rows": len(sample), "excluded_mean_growth": sample.forward_log_growth_raw_30.mean(), "excluded_loss_probability": float((sample.forward_log_growth_raw_30 < math.log(.9)).mean()) if len(sample) else np.nan})
pd.DataFrame(null_rows).to_csv(ARTIFACTS / "veto-permutation-null.csv", index=False)
diagnostic_loss_metrics = []
for horizon in (14, 45):
    diagnostic = walk_forward_loss_predictions(
        features, labels, feature_columns=core, horizon=horizon,
        evaluation_dates=pd.DatetimeIndex(sorted(evaluation.date.unique())),
        min_train_dates=60, c=1.0,
    )
    diagnostic.to_parquet(ARTIFACTS / f"loss-predictions-{horizon}.parquet", index=False)
    diagnostic_loss_metrics.append({"horizon": horizon, **classification_metrics(diagnostic)})
pd.DataFrame(diagnostic_loss_metrics).to_csv(ARTIFACTS / "loss-classifier-horizon-sensitivities.csv", index=False)


,rows,dates,prevalence,brier,log_loss,base_rate_brier,base_rate_log_loss,pr_auc,roc_auc
0,77866,335,0.230768,0.172565,0.562129,0.177514,0.540203,0.396549,0.65848


In [4]:
base = evaluation.copy()
a1_veto_score = base.interval_downside_dev_30.copy()
zero_measured_downside = base.observed_interval_count_30.ge(3) & a1_veto_score.isna()
a1_veto_score.loc[zero_measured_downside] = 0.0
policy_inputs = {
    "A0": (None, None),
    "A1": (base[["date", "address"]].assign(veto_score=a1_veto_score), "veto_score"),
    "A2": (predictions[["date", "address", "loss_probability"]].rename(columns={"loss_probability": "veto_score"}), "veto_score"),
    "A3": (base[["date", "address"]].assign(veto_score=base.btc_beta_30.abs()), "veto_score"),
}
curves, trades, pools, rows = {}, {}, {}, []
for name, (scores, veto_column) in policy_inputs.items():
    curve, trade, pool = simulate_stable_policy(base, observations, policy_name=name, veto_scores=scores, config=CONFIG, max_positions=6, veto_feature=veto_column, veto_fraction=.20)
    curves[name], trades[name], pools[name] = curve, trade, pool
    curve.to_parquet(ARTIFACTS / f"equity-{name}.parquet", index=False)
    trade.to_parquet(ARTIFACTS / f"trades-{name}.parquet", index=False)
    pool.to_parquet(ARTIFACTS / f"pool-{name}.parquet", index=False)
    row = summarise_backtest(curve)
    row.update({"arm": name, "veto_rows": int(pool.vetoed.sum()) if not pool.empty else 0, "veto_active_dates": int(trade.veto_active.sum()) if not trade.empty else 0})
    rows.append(row)
metrics = pd.DataFrame(rows).set_index("arm")
metrics.to_csv(ARTIFACTS / "primary-policy-metrics.csv")
pd.concat({name: curve.set_index("date").equity for name, curve in curves.items()}, axis=1).to_parquet(ARTIFACTS / "primary-equity-comparison.parquet")
display(metrics)


,start,end,final_equity,cagr,volatility,sharpe,max_drawdown,mean_positions,mean_turnover,mean_cash_fraction,mean_stale_positions,mean_carried_positions,veto_rows,veto_active_dates
arm,,,,,,,,,,,,,,
A0,2025-09-13 00:00:00,2026-09-12 00:00:00,157402.271889,0.049487,0.200354,0.340885,-0.156379,6.0,26806.186821,0.024776,0.0,1.972603,0,0
A1,2025-09-13 00:00:00,2026-09-12 00:00:00,154317.387839,0.028863,0.162362,0.256041,-0.085517,6.0,25849.903879,0.027761,0.0,1.967123,7721,183
A2,2025-09-13 00:00:00,2026-09-12 00:00:00,139537.088953,-0.069938,0.161353,-0.368957,-0.149066,6.0,27338.066772,0.029058,0.0,1.972603,7724,183
A3,2025-09-13 00:00:00,2026-09-12 00:00:00,144455.249735,-0.037065,0.166777,-0.143275,-0.103255,6.0,27492.694629,0.021666,0.0,1.969863,5411,117


In [5]:
old_curve, old_trades = simulate_allocator(base, pd.DataFrame(columns=["date", "address", "prediction"]), observations=observations, config=CONFIG, mode="incumbent", max_positions=6)
new_curve = curves["A0"].set_index("date").reindex(old_curve.date).reset_index()
parity = {"rows_old": int(len(old_curve)), "rows_new": int(len(new_curve)), "max_abs_equity_difference": float(np.nanmax(np.abs(old_curve.equity.to_numpy() - new_curve.equity.to_numpy()))), "max_abs_cash_difference": float(np.nanmax(np.abs(old_curve.cash.to_numpy() - new_curve.cash.to_numpy()))), "old_trades": int(len(old_trades)), "new_trades": int(len(trades["A0"])), "same_dates": bool(old_curve.date.equals(new_curve.date))}
write_json(ARTIFACTS / "no-op-parity.json", parity)
print(parity)
assert parity["max_abs_equity_difference"] < 1e-6
assert parity["max_abs_cash_difference"] < 1e-6


{'rows_old': 365, 'rows_new': 365, 'max_abs_equity_difference': 0.0, 'max_abs_cash_difference': 0.0, 'old_trades': 183, 'new_trades': 183, 'same_dates': True}
